In [68]:
import requests
import pandas as pd
import folium

from pyproj import Transformer
from shapely.geometry import LineString, Point

# ============================================================
# 1. CONFIGURATION
# ============================================================

START = "US-46, Columbia, NJ, USA"
FINISH = "US-50, Montrose, CO, USA"

CSV_PATH = "../data/fuel_prices.csv"
OSRM_URL = "https://router.project-osrm.org/route/v1/driving"

MAX_RANGE_MILES = 2000
MPG = 10

# Maximum perpendicular distance from route to station.
CORRIDOR_KM = 25

In [ ]:
# ============================================================
# 2. LOAD ALL GEOCODED STATIONS FROM CSV
# ============================================================

stations = pd.read_csv(CSV_PATH)

required_columns = [
    "Address", "City", "State",
    "Retail Price", "latitude", "longitude"
]

missing_columns = [
    col for col in required_columns
    if col not in stations.columns
]

if missing_columns:
    raise ValueError(f"Missing CSV columns: {missing_columns}")

for col in ["latitude", "longitude", "Retail Price"]:
    stations[col] = pd.to_numeric(
        stations[col], errors="coerce"
    )

stations = stations.dropna(
    subset=["latitude", "longitude", "Retail Price"]
).copy()

# Basic coordinate and price validation
stations = stations[
    stations["latitude"].between(-90, 90)
    & stations["longitude"].between(-180, 180)
    & (stations["Retail Price"] > 0)
].copy()

stations.reset_index(drop=True, inplace=True)

if stations.empty:
    raise ValueError(
        "No stations have valid coordinates and prices. "
        "Geocode more records before running this cell."
    )

print(f"Geocoded station records: {len(stations)}")

Geocoded station records: 122


In [70]:
# ============================================================
# 3. GEOCODE START AND FINISH
# ============================================================

def get_coordinates(place):
    # Reuse the geocode_cached function from your first cell.
    result = geocode_cached(place)

    if result is None:
        raise ValueError(f"Could not locate: {place}")

    return result  # latitude, longitude


start_lat, start_lon = get_coordinates(START)
end_lat, end_lon = get_coordinates(FINISH)

print(f"Coordinates: {start_lat}, {start_lon} -> {end_lat}, {end_lon}")

Coordinates: 40.9228648, -75.0899532 -> 38.5171165, -107.9199568


In [71]:
# ============================================================
# 4. GET DRIVING ROUTE FROM OSRM
# ============================================================

response = requests.get(
    f"{OSRM_URL}/{start_lon},{start_lat};{end_lon},{end_lat}",
    params={
        "overview": "full",
        "geometries": "geojson"
    },
    timeout=60
)

response.raise_for_status()
route_data = response.json()

if route_data.get("code") != "Ok" or not route_data.get("routes"):
    raise RuntimeError(f"OSRM routing failed: {route_data}")

route = route_data["routes"][0]
route_coords = route["geometry"]["coordinates"]

route_miles = route["distance"] / 1609.344
route_hours = route["duration"] / 3600

print(f"\nRoute: {START} -> {FINISH}")
print(f"Driving distance: {route_miles:.1f} miles")
print(f"Estimated driving duration: {route_hours:.2f} hours")

minimum_legs = int(
    -(-route_miles // MAX_RANGE_MILES)
)

print(f"Minimum driving legs: {minimum_legs}")
print(f"Minimum intermediate stops: {max(0, minimum_legs - 1)}")


Route: US-46, Columbia, NJ, USA -> US-50, Montrose, CO, USA
Driving distance: 2003.8 miles
Estimated driving duration: 36.24 hours
Minimum driving legs: 2
Minimum intermediate stops: 1


In [72]:
# ============================================================
# 5. PROJECT ROUTE AND STATION COORDINATES
# ============================================================

# EPSG:5070 is a CONUS projection.
# For this prototype, it is suitable for routes in the
# contiguous United States, not Alaska or Hawaii.
project = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:5070",
    always_xy=True
).transform

route_projected = LineString([
    project(lon, lat)
    for lon, lat in route_coords
])

def project_station(row):
    return Point(
        project(row["longitude"], row["latitude"])
    )

stations["point"] = stations.apply(
    project_station, axis=1
)

stations["distance_from_route_km"] = stations["point"].apply(
    lambda point: route_projected.distance(point) / 1000
)

stations["route_miles"] = stations["point"].apply(
    lambda point: route_projected.project(point) / 1609.344
)

# Only stations in the route corridor
nearby = stations[
    stations["distance_from_route_km"] <= CORRIDOR_KM
].copy()

# Remove duplicate physical descriptions where appropriate.
# Do not deduplicate solely by station ID because IDs may repeat.
nearby = nearby.drop_duplicates(
    subset=[
        "Address", "City", "State",
        "latitude", "longitude", "Retail Price"
    ]
)

nearby = nearby.sort_values(
    "route_miles"
).reset_index(drop=True)

print(f"\nStations within {CORRIDOR_KM} km of route: {len(nearby)}")

display(nearby[
    [
        "Address", "City", "State", "Retail Price",
        "route_miles", "distance_from_route_km"
    ]
])


Stations within 25 km of route: 11


,Address,City,State,Retail Price,route_miles,distance_from_route_km
0,US-46,Columbia,NJ,3.079000,0.338911,0.000008
1,I-94 EXIT 16,Bridgman,MI,3.559000,641.021011,23.076399
2,US-30,Wanatah,IN,3.499000,671.118338,16.195667
3,US-30 & SR-49,Valparaiso,IN,3.869000,674.581795,11.918663
4,US-30 & SR-49,Valparaiso,IN,3.859000,674.581795,11.918663
5,"I-94, EXIT 5",Hammond,IN,3.999000,698.826412,0.022996
6,US-41 & SR-312,Hammond,IN,3.765667,701.231041,4.471585
7,"I-235, EXIT 11",Des Moines,IA,3.027333,1032.738156,5.966732
8,US-30,Grand Island,NE,3.307333,1308.501286,10.288531
9,US-50,Montrose,CO,3.424000,1993.248909,5.181630


In [73]:
# ============================================================
# 6. BUILD CANDIDATE STATIONS
# ============================================================

# Exclude stations almost at the start or destination.
candidates = nearby[
    (nearby["route_miles"] > 1)
    & (nearby["route_miles"] < route_miles - 1)
].copy()

candidates = candidates.sort_values(
    "route_miles"
).reset_index(drop=True)

positions = candidates["route_miles"].tolist()
prices = candidates["Retail Price"].tolist()

print(f"\nIntermediate candidate stations: {len(candidates)}")


Intermediate candidate stations: 10


In [74]:
# ============================================================
# 7. FIND A FEASIBLE SEQUENCE OF FUEL STOPS
# ============================================================

# A station is feasible if the destination can be reached from it,
# directly or via other candidate stations, without exceeding
# the maximum range on any leg.

n = len(candidates)
can_finish = [False] * n

# Base case: can reach the destination from this station.
for i in range(n):
    can_finish[i] = (
        0 <= route_miles - positions[i] <= MAX_RANGE_MILES
    )

# Work backwards to find stations that can eventually reach
# the destination through other stations.
for i in range(n - 1, -1, -1):
    if can_finish[i]:
        continue

    for j in range(i + 1, n):
        gap = positions[j] - positions[i]

        if gap > MAX_RANGE_MILES:
            break

        if gap > 0 and can_finish[j]:
            can_finish[i] = True
            break

# Greedy cost selection among feasible next stops.
selected_indices = []
current_position = 0.0
route_feasible = True

while route_miles - current_position > MAX_RANGE_MILES:
    possible = [
        i for i, position in enumerate(positions)
        if (
            current_position < position <= current_position + MAX_RANGE_MILES
            and can_finish[i]
            and i not in selected_indices
        )
    ]

    if not possible:
        route_feasible = False

        print(
            f"\nNO FEASIBLE STOP: Cannot reach a feasible station "
            f"from mile {current_position:.1f} within "
            f"{MAX_RANGE_MILES} miles."
        )
        break

    # Cheapest reachable station that still permits completion.
    best = min(possible, key=lambda i: prices[i])

    selected_indices.append(best)
    current_position = positions[best]

selected_stops = candidates.iloc[selected_indices].copy()

print("\nRecommended fuel stops:")

if selected_stops.empty:
    print("No intermediate fuel stops could be selected.")
else:
    display(selected_stops[
        [
            "Address", "City", "State", "Retail Price",
            "route_miles", "distance_from_route_km"
        ]
    ])

if route_feasible:
    print("A feasible stop sequence was found.")
else:
    print(
        "The route cannot be completed with the currently "
        "available candidate stations. More station coordinates "
        "are needed."
    )


Recommended fuel stops:


,Address,City,State,Retail Price,route_miles,distance_from_route_km
6,"I-235, EXIT 11",Des Moines,IA,3.027333,1032.738156,5.966732


A feasible stop sequence was found.


In [75]:
# ============================================================
# 8. ESTIMATE FUEL COST
# ============================================================

# This estimate assumes a full tank at the start and purchases
# only the fuel consumed on each route leg.
# It uses the selected stop's price for the leg starting there.
# This is a simplified model; the first leg's fuel price is
# unknown unless the starting location is itself a fuel station.

if route_feasible:
    stop_positions = selected_stops["route_miles"].tolist()
    stop_prices = selected_stops["Retail Price"].tolist()

    leg_boundaries = [0.0] + stop_positions + [route_miles]
    leg_distances = [
        leg_boundaries[i + 1] - leg_boundaries[i]
        for i in range(len(leg_boundaries) - 1)
    ]

    print("\nFuel-leg estimates:")

    total_gallons = 0.0
    estimated_cost = 0.0

    for i, distance in enumerate(leg_distances):
        gallons = distance / MPG
        total_gallons += gallons

        if i == 0:
            price = None
            print(
                f"Start -> first stop: {distance:.1f} miles, "
                f"{gallons:.2f} gallons; starting fuel cost "
                "not included because the starting fuel price is unknown."
            )
        else:
            price = stop_prices[i - 1]
            leg_cost = gallons * price
            estimated_cost += leg_cost

            print(
                f"Leg {i}: {distance:.1f} miles, "
                f"{gallons:.2f} gallons x ${price:.3f} "
                f"= ${leg_cost:.2f}"
            )

    print(f"\nTotal fuel consumed: {total_gallons:.2f} gallons")
    print(
        f"Estimated cost for legs after the first stop: "
        f"${estimated_cost:.2f}"
    )
    print(
        "Note: Total trip cost requires a price for fuel bought "
        "at the starting location or an explicit initial fuel assumption."
    )


Fuel-leg estimates:
Start -> first stop: 1032.7 miles, 103.27 gallons; starting fuel cost not included because the starting fuel price is unknown.
Leg 1: 971.1 miles, 97.11 gallons x $3.027 = $293.97

Total fuel consumed: 200.38 gallons
Estimated cost for legs after the first stop: $293.97
Note: Total trip cost requires a price for fuel bought at the starting location or an explicit initial fuel assumption.


In [76]:
# ============================================================
# 9. CREATE INTERACTIVE ROUTE MAP
# ============================================================

fuel_map = folium.Map(
    location=[start_lat, start_lon],
    zoom_start=5,
    tiles="OpenStreetMap"
)

# Driving route
folium.PolyLine(
    locations=[(lat, lon) for lon, lat in route_coords],
    color="blue",
    weight=5,
    opacity=0.8,
    tooltip=f"Route: {route_miles:.1f} miles"
).add_to(fuel_map)

# Start
folium.Marker(
    [start_lat, start_lon],
    tooltip="Start",
    popup=START,
    icon=folium.Icon(color="green", icon="play")
).add_to(fuel_map)

# Destination
folium.Marker(
    [end_lat, end_lon],
    tooltip="Destination",
    popup=FINISH,
    icon=folium.Icon(color="red", icon="flag")
).add_to(fuel_map)

# Candidate stations: gray
for _, row in candidates.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color="gray",
        fill=True,
        fill_opacity=0.7,
        tooltip=(
            f"{row['City']}, {row['State']} | "
            f"${row['Retail Price']:.3f}/gal"
        )
    ).add_to(fuel_map)

# Selected stops: orange
for _, row in selected_stops.iterrows():
    folium.Marker(
        [row["latitude"], row["longitude"]],
        tooltip="Recommended fuel stop",
        popup=(
            f"{row['Address']}, {row['City']}, {row['State']}<br>"
            f"Price: ${row['Retail Price']:.3f}/gallon<br>"
            f"Route mile: {row['route_miles']:.1f}"
        ),
        icon=folium.Icon(color="orange", icon="tint")
    ).add_to(fuel_map)

fuel_map.fit_bounds([
    [min(lat for lon, lat in route_coords),
     min(lon for lon, lat in route_coords)],
    [max(lat for lon, lat in route_coords),
     max(lon for lon, lat in route_coords)]
])

fuel_map.save("fuel_route_map.html")
print("Map saved to fuel_route_map.html")

Map saved to fuel_route_map.html


In [ ]:
# ============================================================
# 9. CREATE INTERACTIVE ROUTE MAP
# ============================================================

fuel_map = folium.Map(
    location=[start_lat, start_lon],
    zoom_start=5,
    tiles="OpenStreetMap"
)

# Driving route
folium.PolyLine(
    locations=[(lat, lon) for lon, lat in route_coords],
    color="blue",
    weight=5,
    opacity=0.8,
    tooltip=f"Route: {route_miles:.1f} miles"
).add_to(fuel_map)

# Start
folium.Marker(
    [start_lat, start_lon],
    tooltip="Start",
    popup=START,
    icon=folium.Icon(color="green", icon="play")
).add_to(fuel_map)

# Destination
folium.Marker(
    [end_lat, end_lon],
    tooltip="Destination",
    popup=FINISH,
    icon=folium.Icon(color="red", icon="flag")
).add_to(fuel_map)

# Candidate stations: gray
for _, row in candidates.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        color="gray",
        fill=True,
        fill_opacity=0.7,
        tooltip=(
            f"{row['City']}, {row['State']} | "
            f"${row['Retail Price']:.3f}/gal"
        )
    ).add_to(fuel_map)

# Selected stops: orange
for _, row in selected_stops.iterrows():
    folium.Marker(
        [row["latitude"], row["longitude"]],
        tooltip="Recommended fuel stop",
        popup=(
            f"{row['Address']}, {row['City']}, {row['State']}<br>"
            f"Price: ${row['Retail Price']:.3f}/gallon<br>"
            f"Route mile: {row['route_miles']:.1f}"
        ),
        icon=folium.Icon(color="orange", icon="tint")
    ).add_to(fuel_map)

fuel_map.fit_bounds([
    [min(lat for lon, lat in route_coords),
     min(lon for lon, lat in route_coords)],
    [max(lat for lon, lat in route_coords),
     max(lon for lon, lat in route_coords)]
])

fuel_map.save("fuel_route_map.html")
print("Map saved to fuel_route_map.html")

Map saved to fuel_route_map.html


In [77]:
from IPython.display import IFrame, display

display(IFrame("fuel_route_map.html", width="50%", height=300))